Fase 3: Búsqueda Local y Optimización (Recocido Simulado y Algoritmo Genético)
En este cuaderno se implementa la planificación de rutas para múltiples entregas (TSP), 
utilizando pre-cálculo con A*, Recocido Simulado (SA) y Algoritmo Genético (GA), 
junto con el análisis de óptimos locales y generación de gráficas de convergencia.


In [ ]:

import sys
import os
import random
import math
import time
import matplotlib.pyplot as plt
import networkx as nx
import osmnx as ox

# Añadir la ruta raíz del proyecto para poder acceder a la carpeta src y sus módulos
sys.path.append(os.path.abspath(os.path.join('..')))
from src import fase2

print("Librerías importadas correctamente para la Fase 3.")

# 1. Cargar el grafo urbano utilizando la infraestructura de la Fase 2
G, G_proyectado = fase2.cargar_grafo_urbano()

# 2. Seleccionar un origen y puntos de entrega aleatorios
nodos = list(G.nodes)
random.seed(42)  # Mantener reproducibilidad
origen = random.choice(nodos)

alcanzables = [n for n in nx.descendants(G, origen) if n != origen]
puntos_entrega = random.sample(alcanzables, min(10, len(alcanzables)))
puntos_entrega = [origen] + [p for p in puntos_entrega if p != origen][:9]
print(f"Puntos de entrega seleccionados para el TSP: {len(puntos_entrega)}")

# 3. Precalcular matriz de distancias usando A*
def calcular_matriz_distancias(G, G_proyectado, nodos_entrega):
    n = len(nodos_entrega)
    matriz = [[0.0] * n for _ in range(n)]
    print(f"Calculando matriz de distancias ({n}x{n}) con A*...")
    for i in range(n):
        for j in range(n):
            if i != j:
                try:
                    ruta = nx.astar_path(G_proyectado, nodos_entrega[i], nodos_entrega[j], weight='length')
                    distancia = sum(
                        nx.get_edge_attributes(G, 'length').get((u, v, 0), 1.0) 
                        for u, v in zip(ruta[:-1], ruta[1:])
                    )
                    matriz[i][j] = distancia
                except (nx.NetworkXNoPath, nx.NodeNotFound):
                    matriz[i][j] = float('inf')
    return matriz

matriz = calcular_matriz_distancias(G, G_proyectado, puntos_entrega)

def evaluar_ruta(ruta, matriz):
    costo = 0.0
    for i in range(len(ruta)):
        u = ruta[i]
        v = ruta[(i + 1) % len(ruta)]
        costo += matriz[u][v]
    return costo

# 4. Implementación de Recocido Simulado (Simulated Annealing)
def recocido_simulado(matriz, temp_inicial=1000.0, enfriamiento=0.995, iteraciones=1500):
    n = len(matriz)
    actual = list(range(n))
    random.shuffle(actual)
    costo_actual = evaluar_ruta(actual, matriz)
    
    mejor = list(actual)
    mejor_costo = costo_actual
    historial = [costo_actual]
    
    T = temp_inicial
    for _ in range(iteraciones):
        i, j = random.sample(range(n), 2)
        vecino = list(actual)
        vecino[i], vecino[j] = vecino[j], vecino[i]
        costo_vecino = evaluar_ruta(vecino, matriz)
        
        delta = costo_vecino - costo_actual
        if delta < 0 or random.random() < math.exp(-delta / max(T, 1e-5)):
            actual = vecino
            costo_actual = costo_vecino
            if costo_actual < mejor_costo:
                mejor = list(actual)
                mejor_costo = costo_actual
        
        historial.append(mejor_costo)
        T *= enfriamiento
    return mejor, mejor_costo, historial

print("\nEjecutando Recocido Simulado (SA)...")
_, costo_sa, hist_sa = recocido_simulado(matriz)
print(f"Mejor costo SA: {costo_sa:.2f} metros")

# 5. Implementación de Algoritmo Genético
def algoritmo_genetico(matriz, tam_poblacion=60, generaciones=250, tasa_mutacion=0.15):
    n = len(matriz)
    poblacion = [list(range(n)) for _ in range(tam_poblacion)]
    for ind in poblacion:
        random.shuffle(ind)
        
    mejor_historico = None
    mejor_costo_historico = float('inf')
    historial = []
    
    for _ in range(generaciones):
        poblacion = sorted(poblacion, key=lambda ind: evaluar_ruta(ind, matriz))
        costo_mejor_actual = evaluar_ruta(poblacion[0], matriz)
        
        if costo_mejor_actual < mejor_costo_historico:
            mejor_costo_historico = costo_mejor_actual
            mejor_historico = list(poblacion[0])
            
        historial.append(mejor_costo_historico)
        
        # Elitismo: conservar los 10 mejores
        nueva_poblacion = poblacion[:10]  
        while len(nueva_poblacion) < tam_poblacion:
            padre1, padre2 = random.sample(poblacion[:30], 2)
            punto = random.randint(1, n - 1)
            hijo = padre1[:punto] + [x for x in padre2 if x not in padre1[:punto]]
            if random.random() < tasa_mutacion:
                i, j = random.sample(range(n), 2)
                hijo[i], hijo[j] = hijo[j], hijo[i]
            nueva_poblacion.append(hijo)
        poblacion = nueva_poblacion
    return mejor_historico, mejor_costo_historico, historial

print("Ejecutando Algoritmo Genético (GA)...")
_, costo_ga, hist_ga = algoritmo_genetico(matriz)
print(f"Mejor costo GA: {costo_ga:.2f} metros")

# 6. Generación de Gráfica de Convergencia directamente en el Notebook
plt.figure(figsize=(10, 5))
plt.plot(hist_sa, label='Recocido Simulado (SA)', color='blue', linewidth=2)
plt.plot(hist_ga, label='Algoritmo Genético (GA)', color='green', linewidth=2)
plt.xlabel('Iteraciones / Generaciones')
plt.ylabel('Costo Total de la Ruta (metros)')
plt.title('Comparativa de Convergencia: SA vs GA (Fase 3)')
plt.legend()
plt.grid(True)
plt.show()
